In [1]:
import pandas as pd
import re

In [2]:
REGION = 'Thessaly'
enc = 'utf-8'

In [3]:
data = pd.read_csv(f'../../data/{REGION}/GR_{REGION}_GRID_Landcover_2001-2022.csv', encoding=enc)
data.columns = data.columns.str.lower()

In [4]:
data.head()

,x,y,2001_01_01_lc_prop1,2001_01_01_lc_prop1_assessment,2001_01_01_lc_prop2,2001_01_01_lc_prop2_assessment,2001_01_01_lc_prop3,2001_01_01_lc_prop3_assessment,2001_01_01_lc_type1,2001_01_01_lc_type2,...,2022_01_01_lc_prop2_assessment,2022_01_01_lc_prop3,2022_01_01_lc_prop3_assessment,2022_01_01_lc_type1,2022_01_01_lc_type2,2022_01_01_lc_type3,2022_01_01_lc_type4,2022_01_01_lc_type5,2022_01_01_lw,2022_01_01_qc
0,22.911771,38.996698,31,77,36,72.0,30,77,12,12,...,93.0,30,93,10,10,1,6,6,2,0
1,22.934878,38.996698,21,67,20,67.0,20,67,8,8,...,89.0,10,89,1,1,7,1,1,2,0
2,22.957989,38.996698,21,69,20,69.0,20,69,8,8,...,90.0,20,90,9,9,4,2,2,2,0
3,22.773111,39.014659,31,77,30,77.0,30,77,10,10,...,80.0,20,80,9,9,4,1,1,2,0
4,22.796222,39.014659,22,72,20,72.0,20,72,9,9,...,77.0,20,77,9,9,4,1,1,2,0


In [5]:
lc_columns = data.columns.tolist()
lc_columns = [x.split('_')[0] for x in lc_columns]

lc_columns
lc_years = set()

for i in range(0, len(lc_columns)):
    try:
        lc_years.add(int(lc_columns[i]))
    except ValueError:
        pass

lc_years = list(lc_years)

In [6]:
dates_ = re.compile(r"[0-9]{4}_[0-9]{2}_[0-9]{2}_")
landcover_datasets_per_year = []

for year in lc_years:
    frame_name = f'landcover_{year}'
    location_columns = ['x', 'y']
    filtered_columns = [col for col in data if col.startswith(f'{year}')]
    columns_to_keep = location_columns + filtered_columns
    locals()[frame_name] = data[columns_to_keep].copy()
    locals()[frame_name].insert(2, 'year', int(year))
    locals()[frame_name] = locals()[frame_name].rename(columns=lambda x: re.sub(dates_,'',x))
    landcover_datasets_per_year.append(locals()[frame_name])

In [7]:
landcover_processed = pd.concat(landcover_datasets_per_year, ignore_index=True)
landcover_processed.reset_index(drop=True, inplace=True)

In [8]:
landcover_processed

,x,y,year,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc
0,22.911771,38.996698,2001,31,77,36,72.0,30,77,12,12,1,6,7,2,0
1,22.934878,38.996698,2001,21,67,20,67.0,20,67,8,8,4,1,1,2,0
2,22.957989,38.996698,2001,21,69,20,69.0,20,69,8,8,4,1,1,2,0
3,22.773111,39.014659,2001,31,77,30,77.0,30,77,10,10,1,6,6,2,0
4,22.796222,39.014659,2001,22,72,20,72.0,20,72,9,9,4,1,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71231,22.149171,40.164532,2022,14,98,10,98.0,10,98,4,4,6,4,4,2,0
71232,22.172278,40.164532,2022,14,93,10,90.0,10,90,4,4,6,4,4,2,0
71233,22.310938,40.164532,2022,14,95,10,98.0,10,98,4,4,6,4,4,2,0
71234,22.172278,40.182498,2022,14,99,10,99.0,10,99,4,4,6,4,4,2,0


In [9]:
for year in landcover_processed['year'].sort_values().unique():
    subset = landcover_processed[landcover_processed['year'] == year]
    subset.to_csv(f'../../data/{REGION}/GR_{REGION}_Landcover_{year}.csv', index=False, encoding=enc)

In [10]:
landcover_processed.to_csv(f'../../data/{REGION}/GR_{REGION}_Landcover_2001-2022_processed.csv', index=False, encoding=enc)